# 03 · Model misspecification, influence, dummies and logistic regression

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukasquaderer-lgtm/Linear-regression/blob/master/notebooks/03_misspecification_and_extensions.ipynb)

**Goal:** run every diagnostic test from the CFA curriculum on real regression output and apply the fixes:
Breusch–Pagan + White SEs, Durbin–Watson / Breusch–Godfrey + Newey–West, VIF, leverage and Cook’s D,
dummy variables, and a logistic regression for loan default.

In [1]:
# Setup: run this cell first (Shift + Enter). Works in Jupyter, VS Code and Google Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide library deprecation notices

# Use the local data folder if it exists, otherwise read the CSVs straight from GitHub (e.g. in Colab)
DATA = next((p for p in ["../data/", "../../data/"] if Path(p).exists()), "https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/data/")
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (8, 4)

# Answer checker: after an exercise, check("01.1") tells you whether your answer is right
import sys
if Path("../checks.py").exists():
    sys.path.insert(0, "..")  # solution notebooks live one folder down
try:
    from checks import check, check_all
except ImportError:  # in Colab: fetch the checker from GitHub
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/notebooks/checks.py", "checks.py")
    from checks import check, check_all
print("Ready. Data folder:", DATA)

Ready. Data folder: ../data/


In [2]:
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

df = pd.read_csv(DATA + "factor_returns.csv")
X = sm.add_constant(df[["mkt_excess", "smb", "hml", "mom"]])
m = sm.OLS(df["fund_excess"], X).fit()

## 1. Conditional heteroskedasticity: Breusch–Pagan test
BP = n × R² from regressing squared residuals on the X’s; compare with χ²(k).

In [3]:
aux = sm.OLS(m.resid ** 2, X).fit()
BP = m.nobs * aux.rsquared
print(f"BP by hand = {BP:.2f}, chi2 critical (k=4) = {stats.chi2.ppf(0.95, 4):.2f}")
lm, lm_p, _, _ = het_breuschpagan(m.resid, X)
print(f"statsmodels: LM = {lm:.2f}, p = {lm_p:.4f}")
print("→ reject homoskedasticity: conditional heteroskedasticity" if lm_p < 0.05 else "→ no evidence")

BP by hand = 25.89, chi2 critical (k=4) = 9.49
statsmodels: LM = 25.89, p = 0.0000
→ reject homoskedasticity: conditional heteroskedasticity


**Fix:** keep the coefficients, replace the standard errors with White (robust) SEs.

In [4]:
robust = sm.OLS(df["fund_excess"], X).fit(cov_type="HC1")
pd.DataFrame({"coef": m.params, "OLS SE": m.bse, "White SE": robust.bse, "OLS t": m.tvalues, "White t": robust.tvalues}).round(3)

,coef,OLS SE,White SE,OLS t,White t
const,0.157,0.160,0.144,0.982,1.093
mkt_excess,1.019,0.049,0.060,20.600,17.121
smb,0.424,0.061,0.066,6.975,6.434
hml,0.346,0.056,0.050,6.200,6.910
mom,-0.003,0.041,0.047,-0.079,-0.070


## 2. Serial correlation: Durbin–Watson and Breusch–Godfrey

In [5]:
print("DW =", round(durbin_watson(m.resid), 3), "(≈2 means none)")
bg_lm, bg_p, _, _ = acorr_breusch_godfrey(m, nlags=4)
print(f"Breusch-Godfrey (4 lags): LM = {bg_lm:.2f}, p = {bg_p:.3f}")
nw = sm.OLS(df["fund_excess"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})   # Newey-West fix
print(nw.bse.round(3))

DW = 1.578 (≈2 means none)
Breusch-Godfrey (4 lags): LM = 15.44, p = 0.004
const         0.143
mkt_excess    0.058
smb           0.068
hml           0.056
mom           0.048
dtype: float64


## 3. Multicollinearity: VIF = 1 / (1 − R²ⱼ)
We add a variable that is almost a copy of the market to see the damage.

In [6]:
d = df.copy()
d["mkt_copy"] = d["mkt_excess"] + np.random.default_rng(1).normal(0, 0.3, len(d))
Xc = sm.add_constant(d[["mkt_excess", "mkt_copy", "smb", "hml"]])
mc = sm.OLS(d["fund_excess"], Xc).fit()
vif = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])], index=Xc.columns[1:])
print("VIF:"); print(vif.round(1))
print(f"R2 = {mc.rsquared:.3f}, F p-value = {mc.f_pvalue:.1e}")
print(mc.summary().tables[1])

VIF:
mkt_excess    186.6
mkt_copy      184.8
smb             1.0
hml             1.1
dtype: float64
R2 = 0.814, F p-value = 4.9e-41
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1444      0.158      0.912      0.364      -0.169       0.458
mkt_excess     1.5592      0.633      2.462      0.015       0.305       2.814
mkt_copy      -0.5355      0.627     -0.854      0.395      -1.778       0.707
smb            0.4212      0.061      6.952      0.000       0.301       0.541
hml            0.3555      0.055      6.505      0.000       0.247       0.464


Classic symptom: high R² and a significant F, but the two market variables have huge SEs. **Fix:** drop one.

## 4. Influential observations: leverage, studentised residuals, Cook’s D
We plant one extreme month to see if the tests catch it.

In [7]:
d = df[["mkt_excess", "stock_excess"]].copy()
d.loc[len(d)] = [-25.0, 20.0]                 # planted point: market crash, stock soars
Xi = sm.add_constant(d["mkt_excess"])
mi = sm.OLS(d["stock_excess"], Xi).fit()
infl = mi.get_influence()
k, n = 1, len(d)
tab = pd.DataFrame({"leverage": infl.hat_matrix_diag, "student_resid": infl.resid_studentized_external, "cooks_d": infl.cooks_distance[0]})
flags = tab[(tab.leverage > 3 * (k + 1) / n) | (tab.cooks_d > 2 * np.sqrt(k / n)) | (tab.student_resid.abs() > stats.t.ppf(0.975, n - k - 2))]
print("Thresholds: leverage >", round(3 * (k + 1) / n, 3), "| Cook's D >", round(2 * np.sqrt(k / n), 3))
print(flags.round(3))
print("beta with planted point:", round(mi.params.iloc[1], 3), "| without:", round(sm.OLS(df["stock_excess"], sm.add_constant(df["mkt_excess"])).fit().params.iloc[1], 3))

Thresholds: leverage > 0.05 | Cook's D > 0.182
     leverage  student_resid  cooks_d
30      0.055          2.321    0.152
60      0.031         -1.989    0.062
69      0.008         -2.033    0.017
96      0.022         -2.343    0.059
120     0.327          8.616   11.163
beta with planted point: 0.634 | without: 1.304


## 5. Dummy variables
Loan purpose has 4 categories → 3 dummies (the dropped one is the base case).

In [8]:
cr = pd.read_csv(DATA + "credit_default.csv")
dummies = pd.get_dummies(cr["purpose"], prefix="p", drop_first=True, dtype=float)
print("Base category:", sorted(cr["purpose"].unique())[0]); dummies.head()

Base category: auto


,p_debt_consolidation,p_home_improvement,p_small_business
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,0.0,1.0,0.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0


## 6. Logistic regression: probability of default
Estimated by maximum likelihood. Coefficients are changes in **log-odds**; `exp(coef)` is the odds ratio.

In [9]:
Xl = sm.add_constant(pd.concat([cr[["income_k", "debt_to_income", "fico", "loan_to_value", "years_employed", "home_owner"]], dummies], axis=1))
logit = sm.Logit(cr["default"], Xl).fit(disp=0)
print(logit.summary())

                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                  800
Model:                          Logit   Df Residuals:                      790
Method:                           MLE   Df Model:                            9
Date:                Sat, 03 Oct 2026   Pseudo R-squ.:                  0.2610
Time:                        16:45:19   Log-Likelihood:                -300.90
converged:                       True   LL-Null:                       -407.16
Covariance Type:            nonrobust   LLR p-value:                 7.867e-41
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    9.9814      1.595      6.257      0.000       6.855      13.108
income_k                -0.0073      0.003     -2.383      0.017      -0.013      -0.001
debt_to_inco

In [10]:
out = pd.DataFrame({"coef": logit.params, "odds_ratio": np.exp(logit.params), "p": logit.pvalues}).round(4)
print(out)
print(f"\nLikelihood ratio test: LR = {logit.llr:.1f}, p = {logit.llr_pvalue:.2e}; pseudo R2 = {logit.prsquared:.3f}")

                        coef  odds_ratio       p
const                 9.9814  21621.0776  0.0000
income_k             -0.0073      0.9927  0.0172
debt_to_income        0.0718      1.0744  0.0000
fico                 -0.0237      0.9766  0.0000
loan_to_value         0.0412      1.0420  0.0000
years_employed       -0.0387      0.9621  0.0886
home_owner           -0.5217      0.5935  0.0120
p_debt_consolidation -0.1999      0.8188  0.4726
p_home_improvement   -0.0146      0.9855  0.9632
p_small_business      0.5265      1.6931  0.1173

Likelihood ratio test: LR = 212.5, p = 7.87e-41; pseudo R2 = 0.261


### Exercise 1

Re-run the Breusch–Pagan test on the **stock** CAPM regression (`stock_excess` on `mkt_excess`). Store the p-value in `bp_p_stock` and `True`/`False` (heteroskedasticity at 5%?) in `hetero_stock`.

In [11]:
# Your code here

In [12]:
check("03.1");   # run me after your code: ✅ correct, ❌ try again, ⬜ not answered yet

⬜ Exercise 03.1: not answered yet. Store your answer in `bp_p_stock`, `hetero_stock`.


### Exercise 2

Interpret the odds ratio on `fico`: by what percentage do the odds of default change for a 10-point higher FICO score? Store the percentage (e.g. −12.3) in `fico_change_pct`.

<details><summary>Hint</summary>

Odds ratio for 10 points = exp(10 × coef).

</details>

In [13]:
# Your code here

In [14]:
check("03.2");   # run me after your code: ✅ correct, ❌ try again, ⬜ not answered yet

⬜ Exercise 03.2: not answered yet. Store your answer in `fico_change_pct`.


### Exercise 3

Predict the default probability for: income 50, DTI 45, FICO 620, LTV 90, 1 year employed, not a home owner, purpose small_business. Store it in `pd_borrower`.

<details><summary>Hint</summary>

Build a one-row DataFrame with exactly the columns of `Xl` (const = 1 and the three dummy columns), then `logit.predict(row)`.

</details>

In [15]:
# Your code here

In [16]:
check("03.3");   # run me after your code: ✅ correct, ❌ try again, ⬜ not answered yet

⬜ Exercise 03.3: not answered yet. Store your answer in `pd_borrower`.


---
## Solutions

In [17]:
ms = sm.OLS(df["stock_excess"], sm.add_constant(df["mkt_excess"])).fit()
bp_p_stock = het_breuschpagan(ms.resid, ms.model.exog)[1]; hetero_stock = bp_p_stock < 0.05
print("1) BP p-value:", round(bp_p_stock, 4), "→", "heteroskedasticity" if hetero_stock else "no evidence of conditional heteroskedasticity")
b = logit.params["fico"]; fico_change_pct = 100 * (np.exp(10 * b) - 1)
print(f"2) 10-point FICO: odds × {np.exp(10*b):.3f} → {fico_change_pct:.1f}% change")
row = pd.DataFrame([dict(const=1, income_k=50, debt_to_income=45, fico=620, loan_to_value=90, years_employed=1, home_owner=0,
                         p_debt_consolidation=0, p_home_improvement=0, p_small_business=1)])[Xl.columns]
pd_borrower = float(logit.predict(row).iloc[0])
print("3) PD =", round(pd_borrower, 3))
check_all("03")

1) BP p-value: 0.1689 → no evidence of conditional heteroskedasticity
2) 10-point FICO: odds × 0.789 → -21.1% change
3) PD = 0.914
✅ Exercise 03.1: correct.
✅ Exercise 03.2: correct.


✅ Exercise 03.3: correct.

3 of 3 exercises correct.
